In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.chdir("../")

In [4]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [5]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [6]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [7]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [8]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412


In [9]:
c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71


In [495]:
d_name = find_match_using_terms(["Fleet Foxes"], cat2idx)
d = cat2idx[d_name]
print(f"{d_name=}")
print(f"Num Rows: {artist_mat[:, d].sum()}")

d_name='Fleet Foxes (spotify:artist:4EVpmkEwrLYEg6jIsiPMIb)'
Num Rows: 12615


In [497]:
e_name = find_match_using_terms(["Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)"], cat2idx)
e = cat2idx[e_name]
print(f"{e_name=}")
print(f"Num Rows: {artist_mat[:, e].sum()}")

e_name='Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)'
Num Rows: 32356


In [498]:
def zero_target_col(X, target_col):
    X = X.copy()
    
    mask = (X.indices == target_col)

    X.data[mask] = 0
    
    X.eliminate_zeros()
    
    return X

In [1356]:
def scale_column(X, target_col, scale):
    X = X.copy()
    
    mask = (X.indices == target_col)

    X.data[mask] *= scale
    
    return X

In [1647]:
from scipy import sparse

X = artist_mat.tocsc()
n_samples, n_items = X.shape

# items_to_use = [d, e]
items_to_use = [a, b]

y = []
for item_id in items_to_use:
    y.append(X[:, item_id].toarray().flatten())
    
X = sparse.vstack([zero_target_col(X, item_id) for item_id in items_to_use]).tocsr()
# X = sparse.vstack([X for item_id in items_to_use]).tocsr()

# for item_id in items_to_use:
#     X = zero_target_col(X, item_id)
    
y = np.concatenate(y)

indicator_weight = None
# indicator_weight = 20

if indicator_weight is not None:
    indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
    indicator[n_samples:, 0] = indicator_weight

    X = sparse.hstack([X, indicator.tocsr()])

In [1648]:
%%time

# lambda_ = 18000
# model = Ridge(alpha=lambda_, 
#               fit_intercept=False, 
#               positive=True
#              )

# model.fit(X, y)
# similarities = model.coef_

# lambda_ = 36000
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
#                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

CPU times: total: 0 ns
Wall time: 0 ns


In [1657]:
lambda_ = 300
# lambda_ = 200

model = Ridge(alpha=lambda_, 
#               fit_intercept=False, 
#               positive=True
             )

model.fit(X, y)
similarities = model.coef_

# lambda_ = 3500
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
#                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

In [1658]:
# remove indicator variable
if indicator_weight is not None:
    similarities = similarities[:-1]
    
assert len(similarities) == n_items

In [1659]:
np.argsort(-similarities).tolist().index(c)

5

In [1660]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'C

In [1480]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'Little May (spotify:artist:0TjAAwE04BeoSeOpJIakYH)',
 'Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'The Kite String Tangle (spotify:artist:3D6cosC5ZOLCpRxt6T3XS7)',
 'Emma Louise (spotify:artist:1A96iePIMNFBjLrjXEl718)',
 'Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)',
 'Dotan (spotify:artist:1cwOthlzLBwN8Imbq7P71H)',
 'Ed Tullett (spotify:artist:5VGsR5wapeJIuRPX26IeGn)',
 'Josh Jenkins (spotify:artist:20nyUQUh2kYdqOauF3y3Pu)',
 'Quiet Arrows (spotify:artist:7KRnRH8bRvoX4ebQwHw2EI)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Arthur Beatrice (spotify:artist:5Rgh778FK9SlQtBEcNtkqZ)',
 'Aidan Hawken (spotify:artist:2v7uKtvs8C3LSxEUXB0rva)',
 'Neulore (spotify:artist:6SLbaDa56f1CZlUNuF0gk3)',
 'Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'This, the Silent War (spotify:artist:3156bBfcG5gFSWrftU8gaR)',
 'Cathedrals (spotify:artist:7HIjKw4rHS0MtmV0cqSlpB)',
 'Ásgeir Trausti (spotify:artist:7fNWySjsD

In [1436]:
from scipy import sparse

X = artist_mat.tocsc()
n_samples, n_items = X.shape

ya = X[:, d].toarray().flatten()
yb = X[:, e].toarray().flatten()
y = np.concatenate([ya, yb])

# X = sparse.vstack([zero_target_col(X, d), zero_target_col(X, e)]).tocsr()

# scale = 20
# X = sparse.vstack([scale_column(zero_target_col(X, d), e, scale), scale_column(zero_target_col(X, e), d, scale)]).tocsr()

X = sparse.vstack([X, X]).tocsr()
X = zero_target_col(X, d)
X = zero_target_col(X, e)

# indicator_weight = None
indicator_weight = 20

if indicator_weight is not None:
    indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
    indicator[n_samples:, 0] = indicator_weight

    X = sparse.hstack([X, indicator.tocsr()])

In [1441]:
%%time

lambda_ = 18000
model = Ridge(alpha=lambda_, 
              fit_intercept=False, 
              positive=True
             )

model.fit(X, y)
similarities = model.coef_

# lambda_ = 36000
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
#                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

CPU times: total: 4.89 s
Wall time: 4.94 s


In [1442]:
# remove indicator variable
if indicator_weight is not None:
    similarities = similarities[:-1]
    
assert len(similarities) == n_items

In [1443]:
np.argsort(-similarities).tolist().index(a)

225

In [1444]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Iron & Wine (spotify:artist:4M5nCE77Qaxayuhp3fVn4V)',
 'Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)',
 'The Head and the Heart (spotify:artist:0n94vC3S9c3mb2HyNAOcjg)',
 'Sufjan Stevens (spotify:artist:4MXUO7sVCaFgFjoTI5ox5c)',
 'José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)',
 'Band of Horses (spotify:artist:0OdUWJ0sBjDrqHygGUXeCF)',
 'City and Colour (spotify:artist:74gcBzlQza1bSfob90yRhR)',
 'The Shins (spotify:artist:4LG4Bs1Gadht7TCrMytQUO)',
 'James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)',
 'The Paper Kites (spotify:artist:79hrYiudVcFyyxyJW0ipTy)',
 'Daughter (spotify:artist:46CitWgnWrvF9t70C2p1Me)',
 'Bright Eyes (spotify:artist:5o206eFLx38glA2bb4zqIU)',
 'Gregory Alan Isakov (spotify:artist:5sXaGoRLSpd7VeyZrLkKwt)',
 'The Lumineers (spotify:artist:16oZKvXb6WkQlVAjwo2Wbg)',
 'The National (spotify:artist:2cCUtGK9sDU2EoElnk0GNB)',
 'The Avett Brothers (spotify:artist:196lKsA13K3keVXMDFK66q)',
 'Lord Huron (spotify:artist:6ltzsmQQbmdoHHbLZ4ZN25)',

In [1102]:
from scipy import sparse

X = artist_mat
n_samples, n_items = X.shape

ya = X[:, d].toarray().flatten()
yb = X[:, e].toarray().flatten()
# y = (ya + yb)/2

# y = (ya.astype(bool) | yb.astype(bool)).astype(float)
y = (ya.astype(bool) & yb.astype(bool)).astype(float)

X = zero_target_col(X, d)
X = zero_target_col(X, e)

indicator_weight = None

# indicator_weight = 1

# indicator_weight = 5e2
# indicator_weight = 8e2
# indicator_weight = 1e3
# indicator_weight = 3e3

if indicator_weight is not None:
    indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
    indicator[n_samples:, 0] = indicator_weight

    X = sparse.hstack([X, indicator.tocsr()])

In [1151]:
%%time

lambda_ = 30000
model = Ridge(alpha=lambda_, 
              fit_intercept=False, 
              positive=True
             )

model.fit(X, y)
similarities = model.coef_

# lambda_ = 23000
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
# #                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

CPU times: total: 2.58 s
Wall time: 2.61 s


In [1152]:
# remove indicator variabl
if indicator_weight is not None:
    similarities = similarities[:-1]
    
assert len(similarities) == n_items

In [1153]:
np.argsort(-similarities).tolist().index(a)

246

In [1154]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Iron & Wine (spotify:artist:4M5nCE77Qaxayuhp3fVn4V)',
 'Sufjan Stevens (spotify:artist:4MXUO7sVCaFgFjoTI5ox5c)',
 'José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)',
 'The Head and the Heart (spotify:artist:0n94vC3S9c3mb2HyNAOcjg)',
 'The Shins (spotify:artist:4LG4Bs1Gadht7TCrMytQUO)',
 'Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)',
 'The Tallest Man On Earth (spotify:artist:2BpAc5eK7Rz5GAwSp9UYXa)',
 'Band of Horses (spotify:artist:0OdUWJ0sBjDrqHygGUXeCF)',
 'The Middle East (spotify:artist:6imbHAlhHrFwtsOgqpeBK2)',
 'Gregory Alan Isakov (spotify:artist:5sXaGoRLSpd7VeyZrLkKwt)',
 'City and Colour (spotify:artist:74gcBzlQza1bSfob90yRhR)',
 'The Paper Kites (spotify:artist:79hrYiudVcFyyxyJW0ipTy)',
 'Bright Eyes (spotify:artist:5o206eFLx38glA2bb4zqIU)',
 'Radical Face (spotify:artist:5EM6xJN2QNk0cL7EEm9HR9)',
 'Lord Huron (spotify:artist:6ltzsmQQbmdoHHbLZ4ZN25)',
 'The Avett Brothers (spotify:artist:196lKsA13K3keVXMDFK66q)',
 'The Postal Service (spotify:artist:5yV1qdnmxyI

In [951]:
from scipy import sparse

X = artist_mat
n_samples, n_items = X.shape

ya = X[:, d].toarray().flatten()
yb = X[:, e].toarray().flatten()
# y = (ya + yb)/2

y = (ya.astype(bool) | yb.astype(bool)).astype(float)
# y = (ya.astype(bool) & yb.astype(bool)).astype(float)

X = zero_target_col(X, d)
X = zero_target_col(X, e)

indicator_weight = None

# indicator_weight = 1

# indicator_weight = 5e2
# indicator_weight = 8e2
# indicator_weight = 1e3
# indicator_weight = 3e3

if indicator_weight is not None:
    indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
    indicator[n_samples:, 0] = indicator_weight

    X = sparse.hstack([X, indicator.tocsr()])

In [952]:
%%time

# lambda_ = 100
# lambda_ = 1000
# lambda_ = 0

# lambda_ = 30000
lambda_ = 40000

model = Ridge(alpha=lambda_, 
#               fit_intercept=False, 
              positive=True
             )

model.fit(X, y)
similarities = model.coef_

# lambda_ = 0
# lambda_ = 10
# lambda_ = 15
# lambda_ = 100
# lambda_ = 300
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
# #                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

CPU times: total: 3.12 s
Wall time: 3.17 s


In [953]:
# remove indicator variabl
if indicator_weight is not None:
    similarities = similarities[:-1]
    
assert len(similarities) == n_items

In [954]:
np.argsort(-similarities).tolist().index(a)

262

In [955]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Iron & Wine (spotify:artist:4M5nCE77Qaxayuhp3fVn4V)',
 'Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)',
 'The Head and the Heart (spotify:artist:0n94vC3S9c3mb2HyNAOcjg)',
 'The Lumineers (spotify:artist:16oZKvXb6WkQlVAjwo2Wbg)',
 'Sufjan Stevens (spotify:artist:4MXUO7sVCaFgFjoTI5ox5c)',
 'José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)',
 'Band of Horses (spotify:artist:0OdUWJ0sBjDrqHygGUXeCF)',
 'City and Colour (spotify:artist:74gcBzlQza1bSfob90yRhR)',
 'James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)',
 'The Shins (spotify:artist:4LG4Bs1Gadht7TCrMytQUO)',
 'Daughter (spotify:artist:46CitWgnWrvF9t70C2p1Me)',
 'The Paper Kites (spotify:artist:79hrYiudVcFyyxyJW0ipTy)',
 'Mumford & Sons (spotify:artist:3gd8FJtBJtkRxdfbTu19U2)',
 'Bright Eyes (spotify:artist:5o206eFLx38glA2bb4zqIU)',
 'Death Cab for Cutie (spotify:artist:0YrtvWJMgSdVrk3SfNjTbx)',
 'The National (spotify:artist:2cCUtGK9sDU2EoElnk0GNB)',
 'Hozier (spotify:artist:2FXC3k01G6Gw61bmprjgqS)',
 'Lord 

In [677]:
((np.argsort(-similarities).tolist().index(c) + 
  np.argsort(-similarities).tolist().index(d) + 
  np.argsort(-similarities).tolist().index(e)))/3

108737.66666666667

In [678]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['The Ghost In You (spotify:artist:6YAwcFEMfoZdWD75Uzxsq4)',
 'Life in Sweatpants (spotify:artist:1gz7b6h7Jmh8Ixn8byJmCQ)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Dancing Years (spotify:artist:68HC2lyDNLVy8rzBAfLaaZ)',
 'The White Raven (spotify:artist:2DnUdcNeCZpstbr98eBImW)',
 'Arthur Beatrice (spotify:artist:5Rgh778FK9SlQtBEcNtkqZ)',
 'The Noises (spotify:artist:7fbMZx1vPckavUosbLGbFM)',
 'Neon Plastix (spotify:artist:1W7fK3RM2bbtBWjAzCRP4a)',
 'A House For Lions (spotify:artist:4nG1wZZOdWBH94lmLWuh9h)',
 'Peter Piek (spotify:artist:43ubj4kxkN6UZcmRu1ZJDW)',
 'The Contenders (spotify:artist:2SfJeCSWSXkOxFcmFYkUGP)',
 'Iida (spotify:artist:4r8kiXYHx1qXVGZFpyn2s1)',
 'The Night VI (spotify:artist:1IC4SiuNm3guN8LCRIEjGn)',
 'Marcum Stewart (spotify:artist:19zxXBlfQBOsA8SsLY8oOL)',
 'The Novachord Restoration Project (spotify:artist:02JYcd9oCz2PnoqtIbuZsu)',
 'Beno (spotify:artist:3wFhHIgmyMQ95Eqny8uTET)',
 'Ourlives (spotify:artist:5ym1d3rNnxq8QFrkoXfCes)',
 'The Me

In [631]:
from scipy import sparse

X = artist_mat.tocsc()
n_samples, n_items = X.shape

ya = X[:, a].toarray().flatten()
yb = X[:, b].toarray().flatten()
y = np.concatenate([ya, yb])

# X = sparse.vstack([zero_target_col(X, a), zero_target_col(X, b)]).tocsr()

X = sparse.vstack([X, X]).tocsr()
X = zero_target_col(X, a)
X = zero_target_col(X, b)

# indicator_weight = None

# indicator_weight = 1

# indicator_weight = 5e2
# indicator_weight = 8e2
# indicator_weight = 1e3
indicator_weight = 3e3

if indicator_weight is not None:
    indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
    indicator[n_samples:, 0] = indicator_weight

    X = sparse.hstack([X, indicator.tocsr()])

In [654]:
%%time

# lambda_ = 100
# lambda_ = 1000
# lambda_ = 0

lambda_ = 100

model = Ridge(alpha=lambda_, 
              fit_intercept=False, 
              positive=True
             )

model.fit(X, y)
similarities = model.coef_

# lambda_ = 0
# lambda_ = 10
# lambda_ = 15
# lambda_ = 100
# lambda_ = 300
# model = LogisticRegression(C=np.inf if lambda_ == 0 else 1/lambda_, 
# #                            fit_intercept=False
#                           )
# model.fit(X, y)
# similarities = model.coef_[0]

CPU times: total: 24.9 s
Wall time: 25.3 s


In [655]:
# remove indicator variable
if indicator_weight is not None:
    similarities = similarities[:-1]
    
assert len(similarities) == n_items

In [656]:
((np.argsort(-similarities).tolist().index(c) + 
  np.argsort(-similarities).tolist().index(d) + 
  np.argsort(-similarities).tolist().index(e)))/3

26911.666666666668

In [650]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Hozier (spotify:artist:2FXC3k01G6Gw61bmprjgqS)',
 'alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)',
 'Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)',
 'Vance Joy (spotify:artist:10exVja0key0uqUkk6LJRT)',
 'Coldplay (spotify:artist:4gzpq5DPGxSnKTe4SA8HAU)',
 'Glass Animals (spotify:artist:4yvcSjfu4PC0CYQyLy4wSq)',
 'Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)',
 'The Lumineers (spotify:artist:16oZKvXb6WkQlVAjwo2Wbg)',
 'Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)',
 'The Head and the Heart (spotify:artist:0n94vC3S9c3mb2HyNAOcjg)',
 'Ed Sheeran (spotify:artist:6eUKZXaKkcviH0Ku9w2n3V)',
 'James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)',
 'Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)',
 'José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)',
 'Lord Huron (spotify:artist:6ltzsmQQbmdoHHbLZ4ZN25)',
 'Milky Chance (spotify:artist:1hzfo8twXdOegF3xireCYs)',
 'James Blake (spotify:artist:53KwLdlmrlCelAZMaLVZqU)',
 'Flume (spotify:artist:6nxWCVXbOlEVRexSbL

In [482]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'MUNA (spotify:artist:6xdRb2GypJ7DqnWAI2mHGn)',
 'AURORA (spotify:artist:1WgXqy2Dd70QQOU7Ay074N)',
 'Sigur Rós (spotify:artist:6UUrUCIZtQeOf8tC0WuzRy)',
 'Tame Impala (spotify:artist:5INjqkS1o8h1imAzPqGZBb)',
 'Lemaitre (spotify:artist:4CTKqs11Zgsv8EZTVzx764)',
 'Ingrid Michaelson (spotify:artist:2vm8GdHyrJh2O2MfbQFYG0)',
 'Kishi Bashi (spotify:artist:3LVPGE5jPPwtbGslx07YR0)',
 'Harry Styles (spotify:artist:6KImCVD70vtIoJWnq6nGn3)',
 'Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)',
 'Joseph (spotify:artist:5Wfvw7rDz7HA6gE2z6QhqO)',
 'Brandi Carlile (spotify:artist:2sG4zTOLvjKG1PSoOyf5Ej)',
 'Rubblebucket (spotify:artist:6xriZDSK3wPXhOoZXr9fzF)',
 'Woodkid (spotify:artist:44TGR1CzjKBxSHsSEy7bi9)',
 'Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Astrid S (spotify:artist:3AVfmawzu83sp94QW7CEGm)',
 'Milky Chance (spotify:artist:1hzfo8twXdOegF3xireCYs)',
 'Fleet

In [411]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Turbonegro (spotify:artist:191rVxQbbZ05wcICUSvLkz)',
 'TIX (spotify:artist:6CawoDDP1IZUSGl4wSJGC9)',
 'Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Clock Oper

In [283]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)',
 'Mapei (spotify:artist:6baWjwY7WiVPCZcW7pqqhz)',
 'Little May (spotify:artist:0TjAAwE04BeoSeOpJIakYH)',
 'Panama (spotify:artist:3W9UldYu0xJcaOAw2SUTDI)',
 'Odessa (spotify:artist:7xtlNrmdLZS2sqIkgWewi1)',
 'London Grammar (spotify:artist:3Bd1cgCjtCI32PYvDC3ynO)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'On An On (spotify:artist:3Topzt1UZCz8GQlP7Zsu0M)',
 'Frances (spotify:artist:4m6VmvHDXWmVdIw6EJGQ86)',
 'Kopecky (spotify:artist:0vRO9bFgOrDoFtLcDHV8b6)',
 'The New Basement Tapes (spotify:artist:2oQpz9DEfhuSbuT8hjhTDK)',
 'Family of the Year (spotify:artist:7zsin6IgVsR1rqSRCNYDwq)',
 'Milky Chance (spotify:artist:1hzfo8twXdOegF3xireCYs)',
 'Whilk & Misky (spotify:artist:6m9a3tDNWDe6bVR2csIjEv)',
 'Who Is Fancy (spotify:artist:5QSx2vpiSchSeCwc0qmfNI)',
 'Cathedrals (spot

In [278]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)',
 'Panama (spotify:artist:3W9UldYu0xJcaOAw2SUTDI)',
 'Mapei (spotify:artist:6baWjwY7WiVPCZcW7pqqhz)',
 'Little May (spotify:artist:0TjAAwE04BeoSeOpJIakYH)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'The Kite String Tangle (spotify:artist:3D6cosC5ZOLCpRxt6T3XS7)',
 'On An On (spotify:artist:3Topzt1UZCz8GQlP7Zsu0M)',
 'Vacationer (spotify:artist:4rs1K6gDzLY5VnCMSC80o7)',
 'Odessa (spotify:artist:7xtlNrmdLZS2sqIkgWewi1)',
 'Cathedrals (spotify:artist:7HIjKw4rHS0MtmV0cqSlpB)',
 'In The Valley Below (spotify:artist:4WQXRya5np83C21wifjNp9)',
 "Bear's Den (spotify:artist:0nJaMZM8paoA5HEUTUXPqi)",
 'Frances (spotify:artist:4m6VmvHDXWmVdIw6EJGQ86)',
 'Broken Bells (spotify:artist:6dgwEwnK0YtDfS9XhRwBTG)',
 'London Grammar (spotify:artist:3Bd1cgCjtCI32PYvDC3ynO)',
 'Nick Mulvey (spot

In [135]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Samaris (spotify:artist:1Xl5hsislt88ijOD3EZsOY)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Josh Jenkins (spotify:artist:20nyUQUh2kYdqOauF3y3Pu)',
 'Quiet Arrows (spotify:artist:7KRnRH8bRvoX4ebQwHw2EI)',
 'Mobley (spotify:artist:6ZGf9hyQy1jpBZFb7nBWcP)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'This, the Silent War (spotify:artist:3156bBfcG5gFSWrftU8gaR)',
 'Hjaltalín (spotify:artist:3NQUtyu0fGDFqYzG5xhslP)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Jaimi Faulkner (spotify:artist:1J6JaPeWiuT4oEa4oiEALf)',
 'Terrible Sons (spotify:artist:3eaJ1prUilN6z7yoFx9u2g)',
 'Josienne Clarke and Ben Walker (spotify:artist:3Vur5nBUAcGQjXZyk32WV1)',
 'Vök (spotify:artis

In [87]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Clock Opera (spotify:artist:1sSUBGud3QgpkiaQ27XDIm)',
 'TIX (spotify:artist:6CawoDDP1IZUSGl4wSJGC9)',
 'Susanne Sund

In [73]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'The Ghost In You (spotify:artist:6YAwcFEMfoZdWD75Uzxsq4)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Apothek (spotify:artist:2XvtWef9Qq1Qo6icwvGoVH)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'The Banner Days (spotify:artist:2mqN90i4ztaAS3EXphkKU3)',
 'Riz MC (spotify:artist:1QQZjdEPl5zzekVMzwaWdF)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 '

In [67]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 'Clock Opera (spotify:artist:1sSUBGud3QgpkiaQ27XDIm)',
 'Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'Emi

In [60]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 'Clock Opera (spotify:artist:1sSUBGud3QgpkiaQ27XDIm)',
 'Emilie Nicolas (spotify:artist:4cXE1g28uYrIaUisUx5cJt

In [45]:
np.argsort(-similarities).tolist().index(c)

6

In [46]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Apothek (spotify:artist:2XvtWef9Qq1Qo6icwvGoVH)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'The Ghost In You (spotify:artist:6YAwcFEMfoZdWD75Uzxsq4)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 'The Banner Days (spotify:artist:2mqN90i4ztaAS3EXphkKU3)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Emilie Nicolas (spotify:artist:4cXE1g28uYrIaUisUx5c

In [34]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'The Ghost In You (spotify:artist:6YAwcFEMfoZdWD75Uzxsq4)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Apothek (spotify:artist:2XvtWef9Qq1Qo6icwvGoVH)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'The Banner Days (spotify:artist:2mqN90i4ztaAS3EXphkKU3)',
 'Riz MC (spotify:artist:1QQZjdEPl5zzekVMzwaWdF)',
 'Death Vessel (spotify:artist:0CTeeoO6hwJmvuYgoTSH9x)',
 '

In [21]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'The White Birch (spotify:artist:3pDnwHQUEXbkNX3cvidKpG)',
 'Clock Opera (spotify:artist:1sSUBGud3QgpkiaQ27XDIm)',
 'Hkeem (spotify:artist:46XcyK8FnyCJJlvYCUwVZH)',
 'Manchester Orchestra, Frightened Rabbit (spotify:artist:1RYQr

In [145]:
from scipy import sparse

X = artist_mat.tocsc()

ya = X[:, a].toarray().flatten()
yb = X[:, b].toarray().flatten()

n_samples = X.shape[0]

X = sparse.vstack([X, X]).tocsr()

indicator = sparse.lil_matrix((2 * n_samples, 1), dtype=X.dtype)
indicator[n_samples:, 0] = 3e3

# Apply SCALING TRICK: Multiply by huge number
# indicator[n_samples:, 0] = self._scaling_factor
X = sparse.hstack([X, indicator.tocsr()])

y = np.concatenate([ya, yb])

n_items = X.shape[1]

other_items_mask = True
for item_id in [a, b]:
    other_items_mask &= np.arange(n_items) != item_id
    
X = X[:, other_items_mask]

IndexError: bool index 1 has shape (295861,) instead of (591722,)

In [ ]:
lambda_ = 100

In [ ]:
%%time

model = Ridge(alpha=lambda_, 
#               fit_intercept=False, 
#               positive=False
             )

model.fit(X, y)

In [ ]:
# remove indicator variable
similarities = np.zeros(n_items-1)
similarities[other_items_mask[:-1]] = model.coef_[:-1]

In [ ]:
np.argsort(-similarities).tolist().index(c)

In [105]:
np.argsort(-similarities).tolist().index(c)

8

In [100]:
np.argsort(-similarities).tolist().index(c)

11

In [95]:
np.argsort(-similarities).tolist().index(c)

10

In [87]:
np.argsort(-similarities).tolist().index(c)

14

In [64]:
np.argsort(-similarities).tolist().index(c)

14

In [62]:
similarities[np.argsort(-similarities)[:10]].round(2)

array([0.07, 0.04, 0.04, 0.04, 0.04, 0.03, 0.03, 0.03, 0.03, 0.03])

In [64]:
np.argsort(-similarities).tolist().index(c)

14

In [63]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Samaris (spotify:artist:1Xl5hsislt88ijOD3EZsOY)',
 'Josh Jenkins (spotify:artist:20nyUQUh2kYdqOauF3y3Pu)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Mobley (spotify:artist:6ZGf9hyQy1jpBZFb7nBWcP)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'This, the Silent War (spotify:artist:3156bBfcG5gFSWrftU8gaR)',
 'Quiet Arrows (spotify:artist:7KRnRH8bRvoX4ebQwHw2EI)',
 'Hjaltalín (spotify:artist:3NQUtyu0fGDFqYzG5xhslP)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Papertwin (spotify:artist:2aJnm9Ulm51OKN9ZxQLboo)',
 'Jaimi Faulkner (spotify:artist:1J6JaPeWiuT4oEa4oiEALf)',
 'Automatic City (spotify:artist:5AZ3sVuK0T4xA7Zhqe

In [41]:
top_k = 20

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarities)[:top_k].tolist()]

top_k_matches

['Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)',
 'Morten Myklebust (spotify:artist:7zFc6IlzgDwyXojYH1GIkI)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Samaris (spotify:artist:1Xl5hsislt88ijOD3EZsOY)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Josh Jenkins (spotify:artist:20nyUQUh2kYdqOauF3y3Pu)',
 'This, the Silent War (spotify:artist:3156bBfcG5gFSWrftU8gaR)',
 'Mobley (spotify:artist:6ZGf9hyQy1jpBZFb7nBWcP)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Pascal Pinon (spotify:artist:6vuHXpySAbiBH0BlWpznYZ)',
 'Farao (spotify:artist:6XIX2G6ZGiVQgMr6SSTMFu)',
 'The Ghost In You (spotify:artist:6YAwcFEMfoZdWD75Uzxsq4)',
 'Quiet Arrows (spotify:artist:7KRnRH8bRvoX4ebQwHw2EI)',
 'Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)',
 'Papertwin (spotify:artist:2aJnm9Ulm51OKN9ZxQLboo)',
 'Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)',
 'Automatic City (spotify:artist:5AZ3sVuK0T4xA7Zhqey

In [14]:
# Initialize and fit the Ridge regression model.
model = Ridge(alpha=lambda_, fit_intercept=fit_intercept, solver='auto', positive=positive)

# Pass the sparse X and the dense y to the model
model.fit(X_train, y_dense)

# The model coefficients are the similarity scores for the *other* items.
# We need to put them back into a full-sized array.
similarities = np.zeros(n_items)
similarities[other_items_mask] = model.coef_

return similarities


In [12]:
lambda_ = optimize_lambda_using_a_to_b_matching(artist_mat, a, b, fast_approximation=True)

lambda_: 1000000
error: 7523.5
lambda_: 100000.0
error: 4981.0
lambda_: 10000.0
error: 497.5
lambda_: 1000.0
error: 499.0


In [10]:
# lambda_ = optimize_lambda_using_a_to_b_matching(artist_mat, a, b, fast_approximation=False)

In [11]:
# check final error for EASE
a_to_b_error_metric(artist_mat, a, b, lambda_)

lambda_: 100
error: 3.0


3.0

In [12]:
temp = 1

In [13]:
# check error for NPMI in comparison
a_to_b_error_metric_npmi(artist_mat, a, b, temp)

temp: 1
error: 903.0


903.0

In [14]:
top_k = 20

In [15]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(artist_mat, a, lambda_)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)',
 'PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)',
 'The Acid (spotify:artist:0bRtSoJSpQdnbB3dWrWprR)',
 'Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)',
 'Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)',
 'The Careful Ones (spotify:artist:1DdAoWvETBUklcJCOISZx1)',
 'Liza Anne (spotify:artist:426VSUSxx9puUYFgp7l7EQ)',
 'Snakadaktal (spotify:artist:0SdEkx5Ai2gl0W7pnhlsfy)',
 'Vök (spotify:artist:7oDTyDfeA2JE2jUZztkBj8)',
 'Seoul (spotify:artist:3e69LorE0YSsEaYY6x9XuG)',
 'Only Real (spotify:artist:5cyHu7tidauRJ9UawaPwG5)',
 'Big Scary (spotify:artist:4mLYW48jy9Pwv6KpT74Evf)',
 'SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)',
 'San Fermin (spotify:artist:7fSnislKgW9Mz0YIqWQmGt)',
 'Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)',
 'Caroline Smith (spotify:artist:47blM5Op3BJODxUJImwdYE)',
 'Neulore (spotify:artist:6SLbaDa56f1CZlUNuF0gk3)',
 'Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)',
 'Ásgeir Trausti (spotif

In [16]:
# using EASE

similarity_scores = calculate_ease_for_item_cg(artist_mat, b, lambda_)
top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Susanne Sundfør (spotify:artist:54KCNI7URCrG6yjQK3Ukow)',
 'Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)',
 'Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)',
 'Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)',
 'Alice Boman (spotify:artist:3WiytRnvoL0kT3oAGl9TCt)',
 'Majical Cloudz (spotify:artist:4BEYBN6NCPrFk3sOLMTby3)',
 'Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)',
 'Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)',
 'Rubblebucket (spotify:artist:6xriZDSK3wPXhOoZXr9fzF)',
 'Ings (spotify:artist:3wJ2NeHF58Im2tojNX8ESR)',
 'Max Jury (spotify:artist:3MuPVbFDynbq9zRTAqjRxi)',
 'SOAK (spotify:artist:4PLsMEk2DCRVlVL2a9aZAv)',
 'Amason (spotify:artist:4cJKxS7uOPhwb5UQ70sYpN)',
 'Joseph (spotify:artist:5Wfvw7rDz7HA6gE2z6QhqO)',
 'MUNA (spotify:artist:6xdRb2GypJ7DqnWAI2mHGn)',
 'Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)',
 'The Dø (spotify:artist:2mcNCn1qbZUQ3J9KHapUxj)',
 'Karpe Diem (spotify:artist:3X23gpg1vPacr0hBARyxtN)',
 'Matrimony (spotify:a

In [17]:
# using normalized pointwise mutual information

similarity_scores = npmi_batch(artist_mat, a, temp)

top_k_matches = [idx2cat[idx] for idx in np.argsort(-similarity_scores)[:top_k].tolist()]

top_k_matches

['Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)',
 'Jennifer Sullivan (spotify:artist:6Pw9rPhhbzGfWM5nomrH3A)',
 'Shields of Faith (spotify:artist:2hr9UmcGXibjVZe8Binikn)',
 'Blackpocket (spotify:artist:03HmbNfPUCurmGnU4NNUhF)',
 'Gluteus Maximus (spotify:artist:0q0nbjiNlgkzLclUx2m79K)',
 'Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)',
 'Auden (spotify:artist:5HMe9gHJrucpCmLNd2iPSF)',
 'Josienne Clarke and Ben Walker (spotify:artist:3Vur5nBUAcGQjXZyk32WV1)',
 'Cy Jack|Duncan Aran (spotify:artist:53ZZVIzwhRIE7rDMVPaqFw)',
 'Adurn (spotify:artist:4hDzPWJ7vNSD2wCEF49w8r)',
 'James O-L (spotify:artist:1zhN3uwQgyxwLVE2piO0AM)',
 'Labrador Labratories (spotify:artist:7ExeTdXwDt70bNROpS4Gr5)',
 'Macoubre (spotify:artist:1Bsx51kT7wbNEJll4wtcCi)',
 'Kristopher James (spotify:artist:13o5y7dYYnsyKpbq162GIf)',
 'The Project Club (spotify:artist:2A7GpAZpK9GLtYvgTHhuCm)',
 'John Gurney (spotify:artist:689fsusZuLMRcbgu9yImOu)',
 'The Violet Jive (spotify:artist:0g0XgdmP4KpcTUiM5xYA2e)',